# Experiment 05 — Lag sufficiency

**Paper reference.** §2.2 / Appendix B.1 (lag order τ).

**Goal.** Compare τ = 1 vs τ = 2 across the relevant feature families. Specifically: II (τ=1) vs III (τ=2), and V (τ=1) vs VI (τ=2). If the τ=2 specifications do not improve MAE meaningfully, the seller's policy is essentially first-order Markovian — which is the simplest claim to make in §3.2.

In [ ]:
import os, sys, warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import negolin_v15 as nv
nv.setup_matplotlib()

DATA_DIR = '/home/claude/data/merged_results'    # <-- edit if needed
OUT_DIR  = './out'
os.makedirs(OUT_DIR, exist_ok=True)

raw = nv.load_run_files(DATA_DIR)
_, turns = nv.preprocess_turns(raw)
POLICIES = list(nv.POLICIES)
SCHEMES  = list(nv.SCHEMES)
print(f'rows={len(turns):,}  policies={POLICIES}  schemes={SCHEMES}')

In [ ]:
# Load the Stage-2 selection produced by 02_main_sweep.ipynb
sel_path = Path(OUT_DIR) / 'selected_linear.csv'
if sel_path.exists():
    SELECTED = pd.read_csv(sel_path)
    print('Loaded selected_linear.csv:'); display(SELECTED.head())
else:
    # Sensible fallback: Feature II + Ridge per (scheme, policy)
    rows = [{'scheme': s, 'policy': p, 'chosen_family': 'II', 'chosen_model': 'Ridge'}
            for s in SCHEMES for p in POLICIES]
    SELECTED = pd.DataFrame(rows)
    print('selected_linear.csv not found -> using fallback (II/Ridge)')

def picked(scheme, policy):
    r = SELECTED.set_index(['scheme','policy']).loc[(scheme, policy)]
    fam = r['chosen_family']
    if fam == 'Base': fam = 'II'    # Base isn't a feature spec -- fall back
    return fam, r['chosen_model']

In [ ]:
PAIRS = [('II', 'III'), ('V', 'VI')]

rows = []
for sch in SCHEMES:
    for pol in POLICIES:
        sub = turns.query('scheme == @sch and policy == @pol')
        for fam in [f for pair in PAIRS for f in pair]:
            meta, X, y, _ = nv.build_design_matrix(sub, fam)
            (_, _, _), (m_n, X_n, y_n) = nv.partition_persist(meta, X, y)
            if len(y_n) < 50: continue
            pred, _ = nv.grouped_cv_predict(m_n, X_n, y_n, 'Ridge')
            m = nv.cv_metrics(pred)
            rows.append({'scheme': sch, 'policy': pol, 'family': fam,
                         'tau': nv.FEATURE_SPECS[fam]['tau'],
                         'mae': m['mae'], 'mae_se': m['mae_se']})
lag_tbl = pd.DataFrame(rows)
lag_tbl.to_csv(os.path.join(OUT_DIR, 'exp05_lag_sufficiency.csv'), index=False)
lag_tbl.head(8).round(4)

## ΔMAE: τ=1 → τ=2 within each pair

In [ ]:
deltas = []
for (a, b) in PAIRS:
    pa = lag_tbl.query('family == @a').set_index(['scheme','policy'])['mae']
    pb = lag_tbl.query('family == @b').set_index(['scheme','policy'])['mae']
    common = pa.index.intersection(pb.index)
    d = pd.DataFrame({
        'pair': f'{a} -> {b}',
        'mae_tau1': pa.loc[common].values,
        'mae_tau2': pb.loc[common].values,
        'delta':    pb.loc[common].values - pa.loc[common].values,
    }, index=common).reset_index()
    deltas.append(d)
delta = pd.concat(deltas, ignore_index=True)

fig, ax = plt.subplots(figsize=(7, 3.5))
for pair, g in delta.groupby('pair'):
    ax.scatter(g['mae_tau1'], g['mae_tau2'], label=pair, alpha=0.7)
lo, hi = delta[['mae_tau1','mae_tau2']].min().min(), delta[['mae_tau1','mae_tau2']].max().max()
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.7)
ax.set_xlabel('MAE @ τ=1'); ax.set_ylabel('MAE @ τ=2')
ax.set_title('Lag-2 vs Lag-1 MAE across all (scheme, policy) cells')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_exp05_lag_scatter.png'))
plt.show()

print('Mean ΔMAE going τ=1 → τ=2:')
print(delta.groupby('pair')['delta'].agg(['mean','median','std']).round(4))

If `delta` is small and roughly centred at zero, **τ=1 is sufficient** for almost all cells. The exception that matters is family VI: it adds Y_{t-1} (concession velocity), which is functionally a τ=2 feature and *does* tend to help — but that's an information channel, not a higher Markov order.